In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    TargetEncoder
)
from sklearn.feature_extraction import FeatureHasher

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import make_column_transformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score


In [ ]:
# Revisit the OKCupid data
df = pd.read_csv("../04_categorical/profiles_revised.csv")
df.info()

In [ ]:
# Stratify on the drinks category because that's what we did in lecture 4
df["drinks"] = df["drinks"].replace(np.nan, "Unknown")
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["drinks"])

In [ ]:
# revisit cardinality
print("Feature     Count")
# Assuming that str is categorical, could be bad assumption
for feat in train.select_dtypes(include="str").columns:
    print(f"{feat:12}{train[feat].nunique()}")

In [ ]:
# let's check the "pets" category
ax = train["pets"].value_counts().plot.bar()
plt.tight_layout()
plt.savefig("../../img/08-pets-dist.png")

## Combining infrequent categories

In [ ]:
# TODO: see how it changes with max_categories
pet_enc = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
one_hot_pets = pet_enc.fit_transform(train[["pets"]])
one_hot_pets

In [ ]:
one_hot_pets.sum(axis=0)

## Feature Hashing
For very high cardinality strings (or other iterables)

In [ ]:
train["speaks"].value_counts()

In [ ]:
# feature hasher needs lists of strings, and can't handle nans
train.fillna({"speaks": ""}, inplace=True)
language_enc = FeatureHasher(n_features=8, input_type="string")
lang_hash = language_enc.fit_transform(train["speaks"].str.split(",")).todense()

In [ ]:
test = [["french (okay)"], ["english (fluently), spanish (poorly)"],["english, python (okay)", "c++", "japanese"], ["a", "b", "c", "d", "1234"]]
language_enc.transform(test).todense()

## Target encoding
For high cardinal features with a relationship to the target

In [ ]:
# income vs education
ax = train.boxplot(column="income", by="education", rot=90, figsize=(12, 4))
ax.set_yscale("log")
# hmmm... not sure if this is at all useful

In [ ]:
edu_encoder = TargetEncoder(target_type="continuous")

# important! Use fit_transform with this type of encoder
edu_feat = edu_encoder.fit_transform(train[["education"]],train["income"])
order = edu_encoder.encodings_[0].argsort()

for o in order:
    print(f"{edu_encoder.categories_[0][o]:35} ${edu_encoder.encodings_[0][o]:,.2f}")


# Principal Component Analysis (PCA) on Iris Dataset

This example shows a well known decomposition technique known as Principal Component Analysis (PCA) on the [Iris dataset](https://en.wikipedia.org/wiki/Iris_flower_data_set).

This dataset is made of 4 features: sepal length, sepal width, petal length, petal width. We use PCA to project this 4 feature space into a 3-dimensional space.

```
# Authors: The scikit-learn developers
# SPDX-License-Identifier: BSD-3-Clause
```

## Loading the Iris dataset

The Iris dataset is directly available as part of scikit-learn. It can be loaded using the :func:`~sklearn.datasets.load_iris` function. With the default parameters, a :class:`~sklearn.utils.Bunch` object is returned, containing the data, the target values, the feature names, and the target names.

## Plot of pairs of features of the Iris dataset

Let's first plot the pairs of features of the Iris dataset.

In [ ]:
import seaborn as sns
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)

# Rename classes using the iris target names
iris.frame["target"] = iris.target_names[iris.target]
_ = sns.pairplot(iris.frame, hue="target")

Each data point on each scatter plot refers to one of the 150 iris flowers in the dataset, with the color indicating their respective type(Setosa, Versicolor, and Virginica).

You can already see a pattern regarding the Setosa type, which is easily identifiable based on its short and wide sepal. Only considering these two dimensions, sepal width and length, there's still overlap between the Versicolor and Virginica types.

The diagonal of the plot shows the distribution of each feature. We observe that the petal width and the petal length are the most discriminant features for the three types.

## Plot a PCA representation
Let's apply a Principal Component Analysis (PCA) to the iris dataset and then plot the irises across the first three principal components. This will allow us to better differentiate among the three types!



In [ ]:
import matplotlib.pyplot as plt

# unused but required import for doing 3d projections with matplotlib < 3.2
import mpl_toolkits.mplot3d  # noqa: F401

from sklearn.decomposition import PCA

fig = plt.figure(1, figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d", elev=-150, azim=110)

pca = PCA()
X_reduced = pca.fit_transform(iris.data)
scatter = ax.scatter(
    X_reduced[:, 0],
    X_reduced[:, 1],
    X_reduced[:, 2],
    c=iris.target,
    s=40,
)

ax.set(
    title="First three principal components",
    xlabel="1st Principal Component",
    ylabel="2nd Principal Component",
    zlabel="3rd Principal Component",
)
ax.xaxis.set_ticklabels([])
ax.yaxis.set_ticklabels([])
ax.zaxis.set_ticklabels([])

# Add a legend
legend1 = ax.legend(
    scatter.legend_elements()[0],
    iris.target_names.tolist(),
    loc="upper right",
    title="Classes",
)
ax.add_artist(legend1)

plt.show()

PCA will create 3 new features that are a linear combination of the 4 original features. In addition, this transformation maximizes the variance. With this transformation, we can identify each species using only the first principal component.


## Charlotte's Extras
The iris demo is great, but I had a few more things to show off.

In [ ]:
import numpy as np

# PCA is eigendecomposition!
X_centred = iris.data - iris.data.mean(axis = 0)
C =  np.cov(X_centred, rowvar=False)
Lambda, Q = np.linalg.eig(C)
print(Lambda)
print(pca.explained_variance_)

# eigenvalues are columns, pca components are rows for some reason
print(Q)
print(pca.components_.T)


In [ ]:
# eigenvalue version - same, just flipped
X_eig = (X_centred @ Q).values
fig, ax = plt.subplots()
ax.scatter(
    X_eig[:, 0],
    X_eig[:, 1],
    c=iris.target
)
ax.set_aspect("equal")
plt.title("First two principal components")
plt.xlabel("1st Principal Component")
plt.ylabel("2nd Principal Component") 

In [ ]:
# Plot the explained variance ratio
plt.figure()
plt.plot(range(1, 5), pca.explained_variance_ratio_)
plt.ylabel("Variance explained")
plt.xlabel("Component")

In [ ]:
# What is that first one?
plt.bar(range(4), pca.components_[0], width=0.2)
plt.xticks(range(4), iris.feature_names, rotation=45)
plt.ylabel("Weight")

## LDA
Linear Discriminant Analysis (LDA) is similar, but maximizes between-class variance instead of overall variance.

This means it relies on knowing the class labels, so it is a supervised learning technique as well as dimensional reduction.

In [ ]:
# What about LDA instead?
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
lda = LinearDiscriminantAnalysis()
X_lda = lda.fit_transform(iris.data, iris.target)
fig, ax = plt.subplots()
ax.scatter(
    X_lda[:, 0],
    X_lda[:, 1],
    c=iris.target
)
ax.set_aspect("equal")

In [ ]:
# Compare PCA and LDA weights
plt.bar(np.arange(4)-0.1, pca.components_[0], width=0.2, label="PCA")
plt.bar(np.arange(4)+0.1, lda.coef_[0] / lda.coef_[0].sum(), width=0.2, label="LDA")
plt.xticks(range(4), iris.feature_names, rotation=45)
plt.ylabel("Weight")
plt.legend()

In [ ]:
# Plot the explained variance ratio
plt.figure()
plt.plot(range(1, 5), pca.explained_variance_ratio_, label="PCA")
plt.plot(range(1, 3), lda.explained_variance_ratio_, "o", label="LDA")
plt.ylabel("Variance explained")
plt.xlabel("Component")
plt.legend()